# 01 EDA - Amazon Electronics Reviews

This notebook covers the Phase 2 exploratory data analysis tasks for the Fake Review Cartel Detector project.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path.cwd().resolve().parent
RAW_DIR = BASE_DIR / 'data' / 'raw'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
AMAZON_RAW_PATH = RAW_DIR / 'amazon_reviews_us_Electronics_v1_00.csv'
AMAZON_CLEAN_PATH = PROCESSED_DIR / 'amazon_clean.csv'
LABELED_DATASET_PATH = RAW_DIR / 'fake reviews dataset.csv'

CHUNKSIZE = 10000

In [ ]:
preview_chunk = pd.read_csv(AMAZON_RAW_PATH, chunksize=CHUNKSIZE).__next__()
preview_chunk.head()

In [ ]:
clean_df = pd.read_csv(AMAZON_CLEAN_PATH, parse_dates=['review_date'])

summary = {
    'shape': clean_df.shape,
    'dtypes': clean_df.dtypes.astype(str).to_dict(),
    'nulls': clean_df.isnull().sum().to_dict(),
    'unique_reviewers': int(clean_df['customer_id'].nunique()),
    'unique_products': int(clean_df['product_id'].nunique()),
    'verified_purchase_ratio': float(clean_df['verified_purchase'].mean()),
}
summary

In [ ]:
rating_counts = clean_df['star_rating'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(rating_counts.index.astype(str), rating_counts.values, color='#ff6b35')
ax.set_title('Rating Distribution')
ax.set_xlabel('Star Rating')
ax.set_ylabel('Review Count')
plt.show()

In [ ]:
reviews_per_user = clean_df['customer_id'].value_counts()
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(reviews_per_user.values, bins=50, log=True, color='#2a9d8f')
ax.set_title('Reviews per User Distribution')
ax.set_xlabel('Reviews per User')
ax.set_ylabel('Frequency (log scale)')
plt.show()

In [ ]:
reviews_per_day = clean_df.groupby(clean_df['review_date'].dt.date).size()
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(pd.to_datetime(reviews_per_day.index), reviews_per_day.values, color='#264653', linewidth=1)
ax.set_title('Reviews per Day')
ax.set_xlabel('Date')
ax.set_ylabel('Review Count')
plt.show()

In [ ]:
top_100_reviewers = clean_df['customer_id'].value_counts().head(100)
top_100_reviewers.head(10)

In [ ]:
labeled_df = pd.read_csv(LABELED_DATASET_PATH)
labeled_df['label'].value_counts()

## EDA Summary

- Cleaned Amazon Electronics dataset contains **3,089,972 reviews** across **2,152,195 unique reviewers** and **185,715 unique products**.
- Rating distribution is heavily skewed toward 5-star reviews: **1,778,980** 5-star reviews versus **357,577** 1-star reviews.
- Verified purchases account for roughly **84.04%** of all cleaned reviews, which is useful as a trust-related feature later in the pipeline.
- Reviewer activity is extremely long-tailed: the most prolific reviewer wrote **234 reviews**, while most users reviewed only a small number of products.
- Daily review volume spans from **1999-06-09** to **2015-08-31**, giving enough temporal spread for burst-pattern analysis.
- The labeled review dataset currently present in `backend/data/raw/` is balanced by class with **20,216 CG** and **20,216 OR** examples. The file name differs from the Cornell folder structure described in the original plan, so we should keep that in mind before Phase 5.